In [ ]:
# Resolve inputs from the repository root or any folder inside it.
from pathlib import Path
REPO = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "src/featureengineering.py").is_file()), None)
assert REPO is not None, "Open/run this notebook from within the project folder"
MAIN_CSV = REPO / "data/processed/BeijingHousingPrices_featured.csv"
RAW_XLSX = REPO / "data/raw/BeijingHousingPrices.xlsx"
YEARBOOK_DIR = REPO / "notebooks/yearbook"
GEOJSON = YEARBOOK_DIR / "beijing_districts.geojson"
OUT = REPO / "outputs/beijing_a3_visualisations"
for p in [MAIN_CSV, RAW_XLSX, YEARBOOK_DIR / "manifest.json", GEOJSON]:
    assert p.exists(), f"Missing input: {p}. See README.md for data preparation."


In [ ]:
import hashlib
import json
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from matplotlib.colors import BoundaryNorm, LinearSegmentedColormap, TwoSlopeNorm
from matplotlib.dates import AutoDateLocator, DateFormatter
from matplotlib.ticker import FixedLocator, NullLocator
from matplotlib.transforms import Bbox, blended_transform_factory
from shapely import STRtree
from shapely.geometry import Point, box, shape
from shapely.ops import unary_union

sys.path.insert(0, str(REPO / "src"))
import bt2102_style as bs
import importlib
importlib.reload(bs)  # Pick up palette edits when Run All reuses an existing kernel.

sys.path.insert(0, str(REPO / "src"))
from featureengineering import CENTER_LAT, CENTER_LNG   # the point distance_to_center_km is measured from

bs.setup()
OUT.mkdir(parents=True, exist_ok=True)
SRC = "Lianjia transactions, 2012-2017"
BASE, END = 2012, 2017
YEARS = list(range(BASE, END + 1))
N_YRS = END - BASE
CENTRE = (CENTER_LNG, CENTER_LAT)
INDEX_BASE = 2015                      # P2a: index base year, the last year before the 2016-17 boom
POLICY = pd.Timestamp("2017-03-17")   # Beijing "3.17" purchase curbs (higher down payments, shorter loans)
MIN_DISTRICT_TRADES = 100              # minimum sales at each endpoint of the district comparison period
pd.set_option("display.width", 160)


def publish(fig, name, source):
    """Add the source line, shrink any text that runs off the figure, keep the axes clear of the title block and
    the source line, then save to OUT and show the live figure."""
    src_t = fig.text(0.04, 0.025, f"Source: {source}", fontsize=10, color=bs.MUTED, va="bottom")
    fig.canvas.draw()
    r = fig.canvas.get_renderer()
    for t in fig.texts:
        while t.get_window_extent(r).x1 > fig.bbox.width * 0.985 and t.get_fontsize() > 8:
            t.set_fontsize(t.get_fontsize() - 0.5)
    H, gap = fig.bbox.height, 6
    for _ in range(3):
        fig.canvas.draw()
        boxes = [a.get_tightbbox(r) for a in fig.axes]
        head = [t.get_window_extent(r).y0 for t in fig.texts if t.get_position()[1] >= 0.8]
        d_top = max(0, max(b.y1 for b in boxes) - (min(head) - gap)) / H if head else 0
        d_bot = max(0, src_t.get_window_extent(r).y1 + gap - min(b.y0 for b in boxes)) / H
        if d_top < 1e-3 and d_bot < 1e-3:
            break
        for a in fig.axes:
            p = a.get_position()
            a.set_position([p.x0, p.y0 + d_bot, p.width, max(p.height - d_bot - d_top, 0.05)])
    fig.savefig(OUT / name, dpi=bs.DPI)
    display(fig)
    plt.close(fig)


def place_labels(ax, xs, ys, labels, styles=None, size=150, fontsize=11.5):
    """Label scatter points without overlaps: try right, left, above, below and the diagonals, keep the first
    spot that clears every point, every existing text and the legend, and stays inside the axes."""
    fig = ax.figure
    fig.canvas.draw()
    r = fig.canvas.get_renderer()
    rad = np.sqrt(size) / 2 * fig.dpi / 72 + 2
    taken = [Bbox.from_extents(px - rad, py - rad, px + rad, py + rad)
             for px, py in ax.transData.transform(np.column_stack([xs, ys]))]
    taken += [t.get_window_extent(r) for t in ax.texts]
    if ax.get_legend():
        taken.append(ax.get_legend().get_window_extent(r))
    frame = ax.get_window_extent(r)
    cands = [(9, 0, "left", "center"), (-9, 0, "right", "center"), (0, 10, "center", "bottom"), (0, -10, "center", "top"),
             (8, 8, "left", "bottom"), (8, -8, "left", "top"), (-8, 8, "right", "bottom"), (-8, -8, "right", "top"),
             (14, 14, "left", "bottom"), (14, -14, "left", "top"), (-14, 14, "right", "bottom"), (-14, -14, "right", "top")]
    styles = styles or [{}] * len(labels)
    for x, y, lab, st in zip(xs, ys, labels, styles):
        for k, (dx, dy, ha, va) in enumerate(cands + [cands[0]]):
            t = ax.annotate(lab, (x, y), xytext=(dx, dy), textcoords="offset points", ha=ha, va=va,
                            fontsize=st.get("fontsize", fontsize), color=st.get("color", bs.INK),
                            fontweight=st.get("fontweight", "normal"), bbox=st.get("bbox"), zorder=7)
            bb = t.get_window_extent(r)
            inside = frame.x0 <= bb.x0 and bb.x1 <= frame.x1 and frame.y0 <= bb.y0 and bb.y1 <= frame.y1
            if k == len(cands) or (inside and not any(bb.overlaps(o) for o in taken)):
                taken.append(bb)
                break
            t.remove()


def padded(lo, hi, frac=0.08, include=()):
    """Axis limits around the data (and any reference values), with a margin."""
    lo, hi = min([lo, *include]), max([hi, *include])
    pad = (hi - lo) * frac or 1
    return lo - pad, hi + pad


def join_names(xs):
    xs = list(xs)
    return xs[0] if len(xs) == 1 else ", ".join(xs[:-1]) + " and " + xs[-1]


def year_ranges(years):
    runs = []
    for y in sorted(years):
        if runs and y == runs[-1][1] + 1:
            runs[-1][1] = y
        else:
            runs.append([y, y])
    return join_names([f"{a}" if a == b else f"{a}–{str(b)[-2:]}" for a, b in runs])


def phrase(lab):
    """Category label as it reads mid-sentence: '<50 m²' and '0–5 km' stay, 'Top floor' -> 'top floor'."""
    return lab if lab[0].isdigit() or lab[0] in "<>" else lab[0].lower() + lab[1:]

In [ ]:
df = pd.read_csv(MAIN_CSV, low_memory=False)
raw = pd.read_excel(RAW_XLSX, usecols=["id", "DOM"])   # DOM was dropped from the featured file
df["id"] = df["id"].astype(str)
raw["id"] = raw["id"].astype(str)
assert not raw.id.duplicated().any()
df = df.drop(columns="DOM", errors="ignore").merge(raw, on="id", how="left", validate="many_to_one")
n_all = len(df)

df = df[df.trade_year.between(BASE, END) & (df.ladderRatio <= 1)].copy()
df["date"] = pd.to_datetime(df.tradeTime)
df["ym"] = df.date.dt.to_period("M")
df["quarter"] = df.date.dt.to_period("Q")

BANDS = ["0–5 km", "5–10 km", "10–15 km", "15–20 km", "20–30 km", "30+ km"]
SIZE_MAP = {"<50": "<50 m²", "50-90": "50–90 m²", "90-140": "90–140 m²", "140+": "140+ m²"}
AGE_MAP = {"0-5": "0–5 yrs", "6-15": "6–15 yrs", "16-30": "16–30 yrs", "30+": "30+ yrs"}
BTYPES = ["Plate", "Plate + tower", "Tower"]
df["band"] = pd.cut(df.distance_to_center_km, [0, 5, 10, 15, 20, 30, 100], labels=BANDS)
df["btype"] = df.buildingType.map({1: "Tower", 3: "Plate + tower", 4: "Plate"})
df["subway_lab"] = df.subway.map({1: "Near subway", 0: "No subway"})
df["elev_lab"] = df.elevator.map({1: "Has elevator", 0: "No elevator"})
df["size_lab"] = df.size_category.map(SIZE_MAP)
df["age_lab"] = df.age_bucket.map(AGE_MAP)
df["floor_lab"] = df.story_type.map({"Low": "Low floor", "Middle": "Middle floor", "High": "High floor", "Top": "Top floor"})
SIZE_ORDER, AGE_ORDER = list(SIZE_MAP.values()), list(AGE_MAP.values())

# Rule 4: DOM is recorded properly from the month after the last month whose median DOM is still 1 day
dom_m = df.groupby("ym").DOM.median()
low = dom_m[dom_m <= 1].index
DOM_START = low.max() + 1 if len(low) else dom_m.dropna().index.min()
assert DOM_START <= pd.Period(f"{END}-01", "M"), "DOM charts need a full END year of DOM data"
DOM_START_TS = DOM_START.to_timestamp()   # same month, as a timestamp for formatting

print(f"{n_all:,} rows in {MAIN_CSV.name}; {len(df):,} after filters ({BASE}–{END}, ladderRatio ≤ 1); "
      f"{df.DOM.notna().mean():.0%} with DOM")
print(f"Rule 4: DOM recorded properly from {DOM_START_TS:%b %Y}")
print("Rule 5: median followers by year:", df.groupby("trade_year").followers.median().to_dict())

In [ ]:
EN = {"110101": "Dongcheng", "110103": "Dongcheng", "110102": "Xicheng", "110104": "Xicheng",
      "110105": "Chaoyang", "110106": "Fengtai", "110107": "Shijingshan", "110108": "Haidian",
      "110109": "Mentougou", "110111": "Fangshan", "110112": "Tongzhou", "110113": "Shunyi",
      "110114": "Changping", "110115": "Daxing", "110116": "Huairou", "110117": "Pinggu",
      "110228": "Miyun", "110229": "Yanqing", "110118": "Miyun", "110119": "Yanqing"}
gj = json.load(open(GEOJSON, encoding="utf-8"))
polys = {}
for f in gj["features"]:
    nm = EN[str(f["properties"]["adcode"])]
    polys[nm] = unary_union([polys[nm], shape(f["geometry"])]) if nm in polys else shape(f["geometry"])
names = list(polys)
tree = STRtree([polys[n] for n in names])
hit = tree.query([Point(x, y) for x, y in zip(df.Lng, df.Lat)], predicate="within")
df["boundary"] = "outside"
df.iloc[hit[0], df.columns.get_loc("boundary")] = [names[i] for i in hit[1]]
check = pd.crosstab(df.district, df.boundary)
check["share_in_top"] = check.max(axis=1) / check.sum(axis=1)
check["top_boundary"] = check.drop(columns="share_in_top").idxmax(axis=1)

import featureengineering
importlib.reload(featureengineering)
from featureengineering import DISTRICT_MAP
DISTRICT = DISTRICT_MAP.copy()
check["notebook_name"] = check.index.map(DISTRICT)
check["featured_csv_name"] = df.groupby("district").district_name.agg(lambda s: s.mode().iat[0])
print(check[["top_boundary", "share_in_top", "notebook_name", "featured_csv_name"]].round(3))
for code, row in check.iterrows():
    allowed = {DISTRICT[code]}
    if row.top_boundary not in allowed:
        import warnings
        warnings.warn(f"Approximate basemap mismatch for code {code}: notebook label {DISTRICT[code]}, most points in {row.top_boundary}. Original labels retained; boundary/coordinate alignment requires review.")
df["dname"] = df.district.map(DISTRICT)

In [ ]:
manifest = json.load(open(YEARBOOK_DIR / "manifest.json", encoding="utf-8"))
for m in manifest:
    ok = hashlib.sha256(open(YEARBOOK_DIR / m["raw_path"], "rb").read()).hexdigest() == m["sha256"]
    print(f"{m['table_id']:>5}  {m['title_en']:<55} hash ok: {ok}")
    assert ok

CN = {"全市": "Beijing", "东城区": "Dongcheng", "西城区": "Xicheng", "朝阳区": "Chaoyang", "丰台区": "Fengtai",
      "石景山区": "Shijingshan", "海淀区": "Haidian", "门头沟区": "Mentougou", "房山区": "Fangshan",
      "通州区": "Tongzhou", "顺义区": "Shunyi", "昌平区": "Changping", "大兴区": "Daxing", "怀柔区": "Huairou",
      "平谷区": "Pinggu", "密云区": "Miyun", "延庆区": "Yanqing", "北京经济技术开发区": "Yizhuang"}


def yearbook(path):
    t = pd.read_csv(path, skiprows=2, encoding="utf-8-sig")
    t = t[t.iloc[:, 0].isin(CN)].copy()
    t.index = t.iloc[:, 0].map(CN)
    return t.iloc[:, 1:].apply(pd.to_numeric)


pop = yearbook(YEARBOOK_DIR / "district_resident_population_2-1.csv")
income = yearbook(YEARBOOK_DIR / "district_per_capita_disposable_income_2-14.csv")
supply = yearbook(YEARBOOK_DIR / "district_commercial_housing_3-20.csv")
LIANJIA = set(df.dname)
supply

In [ ]:
VOL_QUARTERS = pd.period_range(f"{DISTRICT_BASE}Q1", f"{DISTRICT_END}Q4", freq="Q")
quarter_counts = district_df.groupby(["dname", "quarter"]).size().unstack().reindex(index=dcagr.index, columns=VOL_QUARTERS)
quarter_ok = quarter_counts.ge(30)
adjacent_ok = quarter_ok & quarter_ok.shift(1, axis=1, fill_value=False)
# Compare identical quarterly changes across all districts, without bridging gaps.
COMMON_CHANGES = adjacent_ok.columns[adjacent_ok.all(axis=0)]
assert len(COMMON_CHANGES) == 8
print("Shared supported change endpoints:", [str(q) for q in COMMON_CHANGES])

def volatility_stats(col, data=district_df, min_n=30):
    q = data.groupby([col, "quarter"], observed=True).price.agg(["median", "size"])
    medians = q["median"].where(q["size"] >= min_n).unstack().reindex(columns=VOL_QUARTERS)
    changes = medians.pct_change(axis=1, fill_method=None) * 100
    common = changes.reindex(columns=COMMON_CHANGES)
    stats = pd.DataFrame({"volatility_pp": common.std(axis=1),
                          "available_changes": changes.count(axis=1), "used_changes": common.count(axis=1)})
    stats.loc[stats.used_changes != len(COMMON_CHANGES), "volatility_pp"] = np.nan
    return stats

def volatility(col):
    return volatility_stats(col).volatility_pp.dropna()

vol_support = volatility_stats("dname").reindex(dcagr.index)
vol_d = vol_support.volatility_pp.dropna()
assert len(vol_d) == 13 and vol_support.used_changes.eq(8).all()
display(vol_support)
vol_vals = {}
for col in ["size_lab", "btype", "age_lab"]:
    vol_vals.update(volatility(col).to_dict())
vol_vals.update(vol_d.to_dict())
city_vol = district_df.groupby("quarter").price.median().reindex(VOL_QUARTERS).pct_change(fill_method=None).reindex(COMMON_CHANGES).std() * 100
vol_groups = [("District", list(vol_d.sort_values().index)), ("Unit size", SIZE_ORDER), ("Building type", BTYPES)]
age_spread = max(vol_vals[a] for a in AGE_ORDER) - min(vol_vals[a] for a in AGE_ORDER)
if age_spread >= 0.5:
    vol_groups.append(("Building age", AGE_ORDER))
print(f"age-band volatility spread {age_spread:.2f} pp -> {'shown' if age_spread >= 0.5 else 'left out (bands are near-identical)'}")
steady = {head: min(labs, key=vol_vals.get) for head, labs in vol_groups}
grouped_dots(vol_groups, vol_vals,
             f"Steadiest prices: {steady['District']}, {steady['Unit size']} homes and {phrase(steady['Building type'])} blocks",
             f"Std. dev. of quarterly price change, {DISTRICT_BASE}–{DISTRICT_END} (pp); lower = steadier",
             "Volatility (pp)", "R1_price_volatility.png", ref=city_vol, ref_label=f"Pooled sample {city_vol:.1f}",
             fmt="{:.1f}", colour_rule=lambda v: bs.NEGATIVE if v > city_vol * 1.15 else (bs.HIGHLIGHT if v < city_vol else bs.CONTEXT),
             src=DISTRICT_SRC + ". Same 8 adjacent quarterly changes for every category; ≥30 sales per quarter. No gaps bridged.")



In [ ]:
rows = []
order = 0
for g_i, (group, labels) in enumerate(vol_groups):
    for lab in labels:
        v = vol_vals[lab]
        status = ("Much more volatile" if v > city_vol * 1.15
                  else "Steadier than pooled" if v < city_vol
                  else "Near pooled")
        for point, val in [("Pooled", city_vol), ("Category", v)]:
            rows.append({"group": group, "group_order": g_i, "category": lab,
                         "sort_order": order, "point": point,
                         "volatility_pp": round(val, 2), "status": status,
                         "pooled_vol_pp": round(city_vol, 2)})
        order += 1

vol_tidy = pd.DataFrame(rows)
vol_tidy.to_csv("risk_volatility_dots.csv", index=False)   # change the path to Visualisations/qn 5 if needed
print(vol_tidy.query("point == 'Category'")[["group", "category", "volatility_pp", "status"]])